In [2]:
model_name = "distilgpt2"
device = "cpu"

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

model

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 10563.95it/s]


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [4]:
import pandas as pd

df = pd.read_csv("./llm.csv")
df

,Description,Patient,Doctor
0,Q. What does abutment of the nerve root mean?,"Hi doctor,I am just wondering what is abutting...",Hi. I have gone through your query with dilige...
1,Q. What should I do to reduce my weight gained...,"Hi doctor, I am a 22-year-old female who was d...",Hi. You have really done well with the hypothy...
2,Q. I have started to get lots of acne on my fa...,Hi doctor! I used to have clear skin but since...,Hi there Acne has multifactorial etiology. Onl...
3,Q. Why do I have uncomfortable feeling between...,"Hello doctor,I am having an uncomfortable feel...",Hello. The popping and discomfort what you fel...
4,Q. My symptoms after intercourse threatns me e...,"Hello doctor,Before two years had sex with a c...",Hello. The HIV test uses a finger prick blood ...
...,...,...,...
256911,Why is hair fall increasing while using Bontre...,I am suffering from excessive hairfall. My doc...,"Hello Dear Thanks for writing to us, we are he..."
256912,Why was I asked to discontinue Androanagen whi...,"Hi Doctor, I have been having severe hair fall...","hello, hair4u is combination of minoxid..."
256913,Can Mintop 5% Lotion be used by women for seve...,Hi..i hav sever hair loss problem so consulted...,HI I have evaluated your query thoroughly you...
256914,Is Minoxin 5% lotion advisable instead of Foli...,"Hi, i am 25 year old girl, i am having massive...",Hello and Welcome to ‘Ask A Doctor’ service.I ...


In [5]:
df = df[:1000]
df

,Description,Patient,Doctor
0,Q. What does abutment of the nerve root mean?,"Hi doctor,I am just wondering what is abutting...",Hi. I have gone through your query with dilige...
1,Q. What should I do to reduce my weight gained...,"Hi doctor, I am a 22-year-old female who was d...",Hi. You have really done well with the hypothy...
2,Q. I have started to get lots of acne on my fa...,Hi doctor! I used to have clear skin but since...,Hi there Acne has multifactorial etiology. Onl...
3,Q. Why do I have uncomfortable feeling between...,"Hello doctor,I am having an uncomfortable feel...",Hello. The popping and discomfort what you fel...
4,Q. My symptoms after intercourse threatns me e...,"Hello doctor,Before two years had sex with a c...",Hello. The HIV test uses a finger prick blood ...
...,...,...,...
995,Q. My lax les is 38 cm with inflamed gastric f...,"Hello doctor, My lax les is 38 cm with inflame...",Hello. Gastritis is an inflammation of stomach...
996,Q. I am suffering from mood swings. Kindly adv...,"Hello doctor,I want to get some information re...",Hello. Let me answer your questions via some b...
997,Q. I am having swollen lymph node in my neck. ...,"Hello doctor, I went to the chiropractor and g...",Hello. I do not think that because of chiropra...
998,Q. How good is Albenza for a raccoon roundworm...,"Hello doctor,I am concerned about a possible r...",Hello. Albendazole 400 mg single star dose is ...


In [6]:
from datasets import Dataset
def map_dataset(batch):
    texts = [
        f"Description: {desc}\nPatient: {p}\nDoctor: {doc}"
        for desc, p, doc in zip(batch["Description"], batch["Patient"], batch["Doctor"])
    ]
    return tokenizer(
        texts,
        truncation=True,
        max_length=512,
        return_overflowing_tokens=True

    )

dataset = Dataset.from_pandas(df)
dataset = dataset.map(
    map_dataset,
    batched=True,
    batch_size=8,
    remove_columns=list(df.columns)
)
dataset = dataset.remove_columns(["overflow_to_sample_mapping"])
dataset = dataset.train_test_split(test_size=0.2)
dataset

Map: 100%|██████████| 1000/1000 [00:00<00:00, 3967.15 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 835
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 209
    })
})

In [7]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)
data_collator

DataCollatorForLanguageModeling(tokenizer=GPT2Tokenizer(name_or_path='distilgpt2', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}), mlm=False, whole_word_mask=False, mlm_probability=0.15, mask_replace_prob=0.8, random_replace_prob=0.1, pad_to_multiple_of=None, return_tensors='pt', seed=None)

In [10]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./output/model",
    eval_strategy="epoch", 
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=10
)

trainer =Trainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=training_args,
    data_collator=data_collator
)

trainer.train()

/strg/Documents/Projects/environment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,No log,2.993436
2,No log,2.757801
3,No log,2.601890
4,No log,2.504932
5,2.770411,2.457248
6,2.770411,2.436167
7,2.770411,2.427845
8,2.770411,2.422415
9,2.770411,2.421838
10,2.288630,2.421833


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.07it/s]
/strg/Documents/Projects/environment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 14.18it/s]
/strg/Documents/Projects/environment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=1050, training_loss=2.5166973949614024, metrics={'train_runtime': 7723.2771, 'train_samples_per_second': 1.081, 'train_steps_per_second': 0.136, 'total_flos': 876491115356160.0, 'train_loss': 2.5166973949614024, 'epoch': 10.0})

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained("./output/model/checkpoint-1050")

prompt = "I got cancer."
inputs = tokenizer(prompt, return_tensors="pt").to("cpu")

inputs

outputs = model.generate(
    inputs.input_ids,
    max_new_tokens=100,
    do_sample=True,
    top_k=50,
    top_p=0.95 
)

output_string = tokenizer.batch_decode(outputs)
output_string

/strg/Documents/Projects/environment/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
